In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil
from datetime import datetime, timedelta

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Functions

In [ ]:
def get_tier(flt_ecnl, dict_tiers):
    if flt_ecnl <= dict_tiers['A1']:
        return 'A1'
    elif flt_ecnl <= dict_tiers['A']:
        return 'A'
    elif flt_ecnl <= dict_tiers['B']:
        return 'B'
    elif flt_ecnl <= dict_tiers['C']:
        return 'C'
    elif flt_ecnl <= dict_tiers['D']:
        return 'D'
    else:
        return 'Decline'

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

flt_factor_24_to_72 = 2.36

# dict tiers
dict_tiers = {
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500, 
}

dict_sort = {
    'A1': 1,
    'A': 2,
    'B': 3,
    'C': 4,
    'D': 5,
    'Decline': 6,
}

dict_map = {
    0: 'No',
    1: 'Yes',
}

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Data dictionary

In [ ]:
# get descriptions
df_dict = pd.read_csv('./data_dictionary.csv')
dict_map_descriptions = dict(zip(df_dict['feature_name'], df_dict['Description']))
dict_map_descriptions['miles_odometer__app'] = 'Vehicle mileage'
dict_map_descriptions['ENG-loan_to_value'] = 'Loan to value'
dict_map_descriptions['fltgrossmonthly__income_sum'] = 'Monthly income'
dict_map_descriptions['ENG-bk'] = 'Open BK'
dict_map_descriptions['ENG-franchise'] = 'Franchise'
dict_map_descriptions['ENG-wtd_avg'] = 'Auto payment history'

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/07_get_predictions/{str_filename}'
df = pd.read_parquet(str_uri)
# set dtm
df['request_datetime'] = pd.to_datetime(df['request_datetime'])
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
# make request month
df['request_month'] = df['request_datetime'].apply(
    lambda x: str(x)[:7],
)
# rm
df = df[df['request_month'] != '2021-07'].copy()
# show
df

#### Get lists

In [ ]:
# get cols in model
list_cols_contribution = [col for col in df.columns if 'contribution' in col]
# get cols in model
list_cols_model = [col.split('_binned')[0] for col in list_cols_contribution]

#### Mask negatives

In [ ]:
# mask
for col in list_cols_model:
    df[col] = df[col].mask(df[col] < 0, np.nan)

#### Mean Gen 13 contribution by credit builder y/n

In [ ]:
# agg
dict_agg = {col: 'mean' for col in list_cols_contribution}
df_tmp = df.groupby(by='has_inst_tag', as_index=False).agg(dict_agg)
df_tmp['has_inst_tag'] = df_tmp['has_inst_tag'].map(dict_map)
df_tmp = df_tmp.T
df_tmp = df_tmp.reset_index()
df_tmp.columns = ['feature', 'No (Contribution)', 'Yes (Contribution)']
df_tmp = df_tmp[df_tmp['feature'] != 'has_inst_tag'].copy()
df_tmp['feature'] = df_tmp['feature'].apply(
    lambda x: x.split('_binned')[0],
)

# show
#df_tmp

#### Get the raw values

In [ ]:
dict_agg = {col: 'mean' for col in list_cols_model}
df_tmp2 = df.groupby(by='has_inst_tag', as_index=False).agg(dict_agg)
df_tmp2['has_inst_tag'] = df_tmp2['has_inst_tag'].map(dict_map)
df_tmp2 = df_tmp2.T
df_tmp2 = df_tmp2.reset_index()
df_tmp2.columns = ['feature', 'No', 'Yes']
df_tmp2 = df_tmp2[df_tmp2['feature'] != 'has_inst_tag'].copy()
# rename
dict_rename = {
    'No': 'No (Raw)',
    'Yes': 'Yes (Raw)',
}
df_tmp2.rename(columns=dict_rename, inplace=True)
# join
df_tmp = pd.merge(
    left=df_tmp,
    right=df_tmp2,
    on='feature',
    how='left',
)
# map
df_tmp['Description'] = df_tmp['feature'].map(dict_map_descriptions)

# get diff and sort
df_tmp['Diff'] = df_tmp['Yes (Contribution)'] - df_tmp['No (Contribution)']
# sort
df_tmp.sort_values(by='Diff', ascending=True, inplace=True)

# save
str_filename = 'df_chime_diff.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

#### Proportion chime by tier for Gen 12

In [ ]:
df_tmp = df[df['request_datetime'] >= '2024-09-01'].copy()
df_tmp = df_tmp.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
})
df_tmp['gen12_ecnl'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd'] * flt_factor_24_to_72
# tier
df_tmp['tier'] = df_tmp['gen12_ecnl'].apply(
    lambda x: get_tier(
        flt_ecnl=x,
        dict_tiers=dict_tiers,
    ),
    
)
# group
df_tmp['count'] = 1
df_tmp = df_tmp.groupby(by='tier', as_index=False).agg({
    'has_inst_tag': 'mean',
    'count': 'sum',
})
# sort
df_tmp['tmp'] = df_tmp['tier'].map(dict_sort)
# sort
df_tmp.sort_values(by='tmp', ascending=True, inplace=True)
df_tmp.drop('tmp', axis=1, inplace=True)
# show
df_tmp

#### Proportion chime by tier for Gen 13

In [ ]:
df_tmp = df[df['request_datetime'] >= '2024-09-01'].copy()
df_tmp = df_tmp.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
})
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72
# tier
df_tmp['tier'] = df_tmp['gen13_ecnl'].apply(
    lambda x: get_tier(
        flt_ecnl=x,
        dict_tiers=dict_tiers,
    ),
    
)
# group
df_tmp['count'] = 1
df_tmp = df_tmp.groupby(by='tier', as_index=False).agg({
    'has_inst_tag': 'mean',
    'count': 'sum',
})
# sort
df_tmp['tmp'] = df_tmp['tier'].map(dict_sort)
# sort
df_tmp.sort_values(by='tmp', ascending=True, inplace=True)
df_tmp.drop('tmp', axis=1, inplace=True)
# show
df_tmp

#### Plot over time

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_month': 'first',
    'has_inst_tag': 'max',
    'co_at_720': 'first',
})
int_nrows = df_tmp.shape[0]
df_tmp['count'] = 1
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'count': 'sum',
    'has_inst_tag': 'mean',
    'co_at_720': 'mean',
})
# show
df_tmp

In [ ]:
# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['count']
a = df_tmp['co_at_720']

# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')

# second y
ax2 = ax.twinx()
ax2.set_ylabel('Total Fundings')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Total Fundings')
# legend
ax2.legend(loc='upper center')

# third y
ax3 = ax.twinx()
ax3.spines['right'].set_position(('outward', 60))  # prevent overlap
ax3.set_ylabel('Charge-Off at 24 Months')
ax3.plot(x[:19], a[:19], linestyle='--', color='red', label='Charge-Off at 24 Months')
# legend
ax3.legend(loc='lower right')

# save
str_filename = 'plt_fundings.png'
str_local_path = f'{str_dirname_output}/{str_filename}'
plt.savefig(str_local_path, bbox_inches='tight')

# show
plt.show()

#### Overlay income

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'has_inst_tag': 'max',
    'fltgrossmonthly__income_sum': 'sum',
})
int_nrows = df_tmp.shape[0]
df_tmp['request_month'] = df_tmp['request_datetime'].apply(
    lambda x: str(x)[:7],
)
df_tmp = df_tmp.groupby('request_month', as_index=False).agg({
    'has_inst_tag': 'mean',
    'fltgrossmonthly__income_sum': 'mean',
})

# graph
x = df_tmp['request_month']
y = df_tmp['has_inst_tag']
z = df_tmp['fltgrossmonthly__income_sum']
# ax
fig, ax = plt.subplots(figsize=(9,5))
str_title = f"""
Proportion Has Credit Builder Tag by Request Month
Funded Accounts Only (N Accounts = {int_nrows})
"""
ax.set_title(str_title)
ax.set_xlabel('Request Month')
ax.set_ylabel('Proportion Has Credit Builder Tag')
ax.plot(x, y, label='Has Tag')
ax.tick_params(axis='x', rotation=90)
ax.legend(loc='upper left')
# second y
ax2 = ax.twinx()
ax2.set_ylabel('Mean Income')
ax2.plot(x, z, linestyle='--', color='lightblue', label='Mean Income')
# legend
ax2.legend(loc='upper center')
# show
plt.show()

#### Maximum date

In [ ]:
# get max date
dtm_max = df['request_datetime'].max()
print(f'Maximum date in data: {dtm_max}')

#### 60 in 720

In [ ]:
int_ndays = 720
str_co_at = f'co_at_{int_ndays}'
# subtract n days
dtm_max_new = dtm_max - timedelta(days=int_ndays)
# make first of month
dtm_max_new = dtm_max_new.replace(day=1)
print(f'Subsetting to < {dtm_max_new}')
# subset
df_tmp = df[df['request_datetime'] < dtm_max_new].copy()
# group
df_tmp = df_tmp.groupby(by='accountid', as_index=False).agg({
    'request_month': 'first',
    'has_inst_tag': 'max',
    'gen12_pd': 'mean',
    'gen12_lgd': 'mean',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    str_co_at: 'mean',
})
# make co at 72
df_tmp[str_co_at] = df_tmp[str_co_at] * flt_factor_24_to_72
# get ecnl
df_tmp['gen12_ecnl'] = df_tmp['gen12_pd'] * df_tmp['gen12_lgd'] * flt_factor_24_to_72
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72

# show
df_tmp

#### Plot by month

In [ ]:
list_cols = [
    'request_month',
    'has_inst_tag',
]
df_tmp['count'] = 1
df_tmp2 = df_tmp.groupby(by=list_cols, as_index=False).agg({
    'count': 'sum',
    'gen12_ecnl': 'sum',
    'gen13_ecnl': 'sum',
    'co_at_720': 'sum',
})
# subset
df_0 = df_tmp2[df_tmp2['has_inst_tag'] == 0].copy()
list_cols = [f'{col}_0' for col in df_0.columns]
df_0.columns = list_cols
# subset
df_1 = df_tmp2[df_tmp2['has_inst_tag'] == 1].copy()
list_cols = [f'{col}_1' for col in df_1.columns]
df_1.columns = list_cols
# concat
df_tmp2 = pd.merge(
    left=df_0,
    right=df_1,
    left_on='request_month_0',
    right_on='request_month_1',
    how='inner',
)
# drop
list_cols = [
    'has_inst_tag_0',
    'request_month_1',
    'has_inst_tag_1',
]
df_tmp2.drop(list_cols, axis=1, inplace=True)
# rename
dict_rename = {
    'request_month_0': 'request_month',
}
df_tmp2.rename(columns=dict_rename, inplace=True)
# cumsums
list_cols = [col for col in df_tmp2.columns if col != 'request_month']
for col in tqdm(list_cols):
    df_tmp2[col] = df_tmp2[col].cumsum()
# cummeans
list_cols = [
    'gen12_ecnl_0',
    'gen13_ecnl_0',
    'co_at_720_0',
]
for col in tqdm(list_cols):
    df_tmp2[col] = df_tmp2[col] / df_tmp2['count_0']
# cummeans
list_cols = [
    'gen12_ecnl_1',
    'gen13_ecnl_1',
    'co_at_720_1',
]
for col in tqdm(list_cols):
    df_tmp2[col] = df_tmp2[col] / df_tmp2['count_1']

# create factors
df_tmp2['gen12_factor_0'] = df_tmp2['co_at_720_0'] / df_tmp2['gen12_ecnl_0']
df_tmp2['gen12_factor_1'] = df_tmp2['co_at_720_1'] / df_tmp2['gen12_ecnl_1']
# create factors
df_tmp2['gen13_factor_0'] = df_tmp2['co_at_720_0'] / df_tmp2['gen13_ecnl_0']
df_tmp2['gen13_factor_1'] = df_tmp2['co_at_720_1'] / df_tmp2['gen13_ecnl_1']

# show
df_tmp2

In [ ]:
list_cols = [
    'gen12_factor_0',
    'gen12_factor_1',
    'gen13_factor_0',
    'gen13_factor_1',
]
x = df_tmp2['request_month']
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Suggested Factors by Model by Credit Builder (1) or Not (0)')
ax.set_ylabel('Factor')
ax.axhline(1, linestyle='--', color='lightblue', label='1.0')
for col in tqdm(list_cols):
    ax.plot(x, df_tmp2[col], label=col)
# xtick labels
ax.set_xticklabels(x, rotation=90)
# legend
ax.legend()
# show
plt.show()